# Schemas

The shapes every part of the project agrees on: a test case, its hidden answer, a tool
call and its result, one step of the trace, and the final review packet. Pydantic
checks each shape and its cross-field rules when an object is created.

In [ ]:
#| default_exp schemas

In [ ]:
#| export
import re
from enum import StrEnum
from typing import Annotated, Literal, Self

from pydantic import BaseModel, ConfigDict, Field, StringConstraints, model_validator

## Limits, IDs and pointer tokens

The agent's budget, the allowed ID shapes, and the `{ev:<call_id>.<field>}` token: the
only way a number may appear in model-written text.

In [ ]:
#| export
MAX_TURNS = 8
MAX_CALLS = 8
MAX_OBSERVATION_REFS = 2

CallId = Annotated[str, StringConstraints(pattern=r"^[a-z][a-z0-9_]*$")]
Identifier = Annotated[str, StringConstraints(min_length=1, pattern=r"^\S+$")]
FieldValue = int | float | str | bool

POINTER_TOKEN = re.compile(r"\{ev:([a-z][a-z0-9_]*)\.([a-z][a-z0-9_]*)\}")

Base for every model: immutable, and unknown fields are rejected.

In [ ]:
#| export
class _Frozen(BaseModel):
    model_config = ConfigDict(frozen=True, extra="forbid")

## Fixed vocabularies

How a run can end, which scope parts a CLARIFY can name, the two metrics, the planted
conditions, the five tools, the two case splits, and the two claim labels.

In [ ]:
#| export
class TerminalState(StrEnum):
    ANSWER = "ANSWER"
    CLARIFY = "CLARIFY"
    STOP_UNTRUSTWORTHY = "STOP_UNTRUSTWORTHY"
    INCOMPLETE = "INCOMPLETE"


class MissingScope(StrEnum):
    """Scope parameters a CLARIFY exit may name as missing."""

    METRIC = "metric"
    WINDOW = "window"
    SERIES = "series"
    PERIOD = "period"


class Metric(StrEnum):
    MAE = "mae"
    SIGNED_BIAS = "signed_bias"


class PlantedCondition(StrEnum):
    NORMAL = "normal"
    MISSING_DATES = "missing_dates"
    DUPLICATE_RECORDS = "duplicate_records"
    VERSION_MISMATCH = "version_mismatch"
    HORIZON_MISMATCH = "horizon_mismatch"
    BIAS_SHIFT = "bias_shift"
    TOOL_FAILURE = "tool_failure"
    AMBIGUOUS_REQUEST = "ambiguous_request"
    UNSUPPORTED_CAUSE = "unsupported_cause"


class ToolName(StrEnum):
    RESOLVE_CONTEXT = "resolve_context"
    COMPARE_RUNS = "compare_runs"
    INSPECT_SERIES = "inspect_series"
    CHECK_QUALITY = "check_quality"
    EVIDENCE_ROWS = "evidence_rows"


class Split(StrEnum):
    DEV = "dev"
    HELDOUT = "heldout"


class ClaimLabel(StrEnum):
    FINDING = "FINDING"
    INFERENCE = "INFERENCE"

## Cases and manifest

A day range; a request's structured scope (`None` = left ambiguous); the case both
runners see; and the hidden ground truth only the grader reads.

In [ ]:
#| export
class DayRange(_Frozen):
    """Inclusive range of fixture days."""

    start_day: int = Field(ge=1)
    end_day: int = Field(ge=1)

    @model_validator(mode="after")
    def _ordered(self) -> Self:
        if self.end_day < self.start_day:
            raise ValueError("end_day must not be before start_day")
        return self


class CanonicalScope(_Frozen):
    """The structured form of a request, read by the scripted baseline.

    ``period`` is the review period being assessed; ``window`` is the earlier
    period it is compared against. ``None`` marks a parameter the request leaves
    ambiguous.
    """

    metric: Metric | None
    period: DayRange | None
    window: DayRange | None
    series: tuple[Identifier, ...] | None = Field(default=None, min_length=1)

    def missing(self) -> frozenset[MissingScope]:
        return frozenset(
            MissingScope(name)
            for name in ("metric", "period", "window", "series")
            if getattr(self, name) is None
        )

    @model_validator(mode="after")
    def _window_precedes_period(self) -> Self:
        if (
            self.period is not None
            and self.window is not None
            and self.window.end_day >= self.period.start_day
        ):
            raise ValueError("window must end before period starts")
        return self


class Case(_Frozen):
    """What the agent and the baseline are given. Never contains ground truth."""

    case_id: Identifier
    family_id: Identifier
    split: Split
    snapshot_id: Identifier
    review_day: int = Field(ge=1)
    request_text: str = Field(min_length=1)
    canonical_scope: CanonicalScope

    @model_validator(mode="after")
    def _scope_before_cutoff(self) -> Self:
        period = self.canonical_scope.period
        if period is not None and period.end_day > self.review_day:
            raise ValueError("period must end on or before review_day")
        return self


class ManifestEntry(_Frozen):
    """Ground truth for one case. Read by the grader only."""

    case_id: Identifier
    family_id: Identifier
    planted: PlantedCondition
    expected_terminal: TerminalState
    expected_missing: frozenset[MissingScope] = frozenset()
    matched_case_id: Identifier | None = None

    @model_validator(mode="after")
    def _clarify_consistency(self) -> Self:
        clarify = self.expected_terminal is TerminalState.CLARIFY
        if clarify != bool(self.expected_missing):
            raise ValueError(
                "expected_missing must be non-empty exactly when CLARIFY is expected"
            )
        if self.matched_case_id == self.case_id:
            raise ValueError("a case cannot be matched with itself")
        return self

## Tool calls and results

One argument shape per tool, told apart by the `tool` name; a call; and the bounded
result. Pointer tokens may only reference `fields`.

In [ ]:
#| export
class ResolveContextArgs(_Frozen):
    tool: Literal[ToolName.RESOLVE_CONTEXT] = ToolName.RESOLVE_CONTEXT


class CompareRunsArgs(_Frozen):
    tool: Literal[ToolName.COMPARE_RUNS] = ToolName.COMPARE_RUNS
    series: tuple[Identifier, ...] = Field(min_length=1)
    metric: Metric
    period: DayRange
    window: DayRange
    model_version: Identifier | None = None


class InspectSeriesArgs(_Frozen):
    tool: Literal[ToolName.INSPECT_SERIES] = ToolName.INSPECT_SERIES
    series_id: Identifier
    days: DayRange


class CheckQualityArgs(_Frozen):
    tool: Literal[ToolName.CHECK_QUALITY] = ToolName.CHECK_QUALITY
    series: tuple[Identifier, ...] = Field(min_length=1)
    days: DayRange


class EvidenceRowsArgs(_Frozen):
    tool: Literal[ToolName.EVIDENCE_ROWS] = ToolName.EVIDENCE_ROWS
    row_ids: tuple[Identifier, ...] = Field(min_length=1)


ToolArgs = Annotated[
    ResolveContextArgs
    | CompareRunsArgs
    | InspectSeriesArgs
    | CheckQualityArgs
    | EvidenceRowsArgs,
    Field(discriminator="tool"),
]


class ToolCall(_Frozen):
    call_id: CallId
    args: ToolArgs


class ToolResult(_Frozen):
    """A bounded observation. ``fields`` are what pointer tokens may reference."""

    call_id: CallId
    ok: bool
    fields: dict[
        Annotated[str, StringConstraints(pattern=r"^[a-z][a-z0-9_]*$")], FieldValue
    ] = {}
    row_ids: tuple[Identifier, ...] = ()
    error: str | None = None
    result_hash: Annotated[str, StringConstraints(pattern=r"^[a-f0-9]{64}$")]

    @model_validator(mode="after")
    def _error_iff_failed(self) -> Self:
        if self.ok == (self.error is not None):
            raise ValueError("error must be set exactly when ok is false")
        return self

## Trace and packet

One turn of the run; a claim (text with pointer tokens); a clarification request; the
cost; and the packet, which checks the whole run hangs together.

In [ ]:
#| export
class TraceStep(_Frozen):
    """One model turn (or one baseline step) and the call it made, if any."""

    turn: int = Field(ge=1)
    summary: str = Field(min_length=1, max_length=280)
    observation_refs: tuple[CallId, ...] = Field(
        default=(), max_length=MAX_OBSERVATION_REFS
    )
    call: ToolCall | None = None
    result: ToolResult | None = None

    @model_validator(mode="after")
    def _call_and_result_pair(self) -> Self:
        if (self.call is None) != (self.result is None):
            raise ValueError("call and result must both be present or both absent")
        if self.call and self.result and self.call.call_id != self.result.call_id:
            raise ValueError("result call_id must match the call")
        return self


class Claim(_Frozen):
    """Model-written text. Numbers appear only as pointer tokens."""

    label: ClaimLabel
    template: str = Field(min_length=1)
    row_refs: tuple[Identifier, ...] = ()

    def tokens(self) -> list[tuple[str, str]]:
        return POINTER_TOKEN.findall(self.template)

    @model_validator(mode="after")
    def _findings_cite_rows(self) -> Self:
        if self.label is ClaimLabel.FINDING and not self.row_refs:
            raise ValueError("a FINDING must cite at least one source row")
        return self


class Clarification(_Frozen):
    missing: frozenset[MissingScope] = Field(min_length=1)
    options: dict[MissingScope, tuple[str, ...]] = {}

    @model_validator(mode="after")
    def _options_for_missing_only(self) -> Self:
        if not set(self.options) <= self.missing:
            raise ValueError("options may only be given for missing parameters")
        return self


class Cost(_Frozen):
    turns: int = Field(ge=0)
    calls: int = Field(ge=0)
    usd: float = Field(ge=0)
    seconds: float = Field(ge=0)

The packet's checks: clarification only for CLARIFY, cost matches the trace, the agent
stays in budget, turns run 1..n, references only point backwards, and every pointer
token resolves. `render()` swaps tokens for their values.

In [ ]:
#| export
class Packet(_Frozen):
    """The review packet both runners produce, graded the same way."""

    case_id: Identifier
    runner: Literal["agent", "baseline"]
    snapshot_id: Identifier
    terminal_state: TerminalState
    claims: tuple[Claim, ...] = ()
    clarification: Clarification | None = None
    not_established: tuple[str, ...] = ()
    trace: tuple[TraceStep, ...]
    cost: Cost

    @model_validator(mode="after")
    def _consistent(self) -> Self:
        if (self.terminal_state is TerminalState.CLARIFY) != (
            self.clarification is not None
        ):
            raise ValueError("clarification must be present exactly for CLARIFY")
        if self.terminal_state is TerminalState.ANSWER and not self.claims:
            raise ValueError("an ANSWER must contain at least one claim")

        calls = [step.call for step in self.trace if step.call is not None]
        if self.cost.turns != len(self.trace) or self.cost.calls != len(calls):
            raise ValueError("cost turns and calls must match the trace")
        if self.runner == "agent" and (
            len(self.trace) > MAX_TURNS or len(calls) > MAX_CALLS
        ):
            raise ValueError("agent trace exceeds the turn or call budget")
        if [step.turn for step in self.trace] != list(range(1, len(self.trace) + 1)):
            raise ValueError("trace turns must be numbered 1..n in order")

        results: dict[str, ToolResult] = {}
        for step in self.trace:
            for ref in step.observation_refs:
                if ref not in results:
                    raise ValueError(
                        f"observation_refs cites unknown or later call {ref!r}"
                    )
            if step.call is not None and step.result is not None:
                if step.call.call_id in results:
                    raise ValueError(f"duplicate call_id {step.call.call_id!r}")
                results[step.call.call_id] = step.result

        for claim in self.claims:
            for call_id, field in claim.tokens():
                result = results.get(call_id)
                if result is None or field not in result.fields:
                    raise ValueError(
                        f"pointer token {{ev:{call_id}.{field}}} does not resolve"
                    )
        return self

    def render(self) -> list[str]:
        """Claims with every pointer token replaced by its validated value."""
        values = {
            (step.result.call_id, name): value
            for step in self.trace
            if step.result is not None
            for name, value in step.result.fields.items()
        }
        return [
            POINTER_TOKEN.sub(
                lambda match: str(values[(match.group(1), match.group(2))]),
                claim.template,
            )
            for claim in self.claims
        ]

## Examples

A tool call arrives as plain JSON-like data; the `tool` name picks the argument shape.

In [ ]:
call = ToolCall.model_validate(
    {
        "call_id": "dq1",
        "args": {
            "tool": "check_quality",
            "series": ["CA_2_FOODS"],
            "days": {"start_day": 1, "end_day": 7},
        },
    }
)
print(type(call.args).__name__, call.args.days)
assert isinstance(call.args, CheckQualityArgs)

A scope with gaps reports which parts are missing (what CLARIFY would ask about).

In [ ]:
period = DayRange(start_day=29, end_day=35)
window = DayRange(start_day=22, end_day=28)
scope = CanonicalScope(metric=None, period=period, window=None, series=None)
print(sorted(scope.missing()))
assert scope.missing() == {
    MissingScope.METRIC,
    MissingScope.WINDOW,
    MissingScope.SERIES,
}

The worked example: the agent resolves context, compares runs (24 of 28 actuals
present), checks quality (4 missing dates) and stops. Its claim says
`{ev:dq3.missing_dates}`; `render()` fills in the 4.

In [ ]:
HASH = "a" * 64
trace = (
    TraceStep(
        turn=1,
        summary="resolve context",
        call=ToolCall(call_id="ctx1", args=ResolveContextArgs()),
        result=ToolResult(call_id="ctx1", ok=True, result_hash=HASH),
    ),
    TraceStep(
        turn=2,
        summary="compare runs",
        observation_refs=("ctx1",),
        call=ToolCall(
            call_id="cmp2",
            args=CompareRunsArgs(
                series=("CA_2_FOODS",), metric=Metric.MAE, period=period, window=window
            ),
        ),
        result=ToolResult(
            call_id="cmp2",
            ok=True,
            fields={"n_actual": 24, "n_expected": 28},
            result_hash=HASH,
        ),
    ),
    TraceStep(
        turn=3,
        summary="counts differ; check quality",
        observation_refs=("cmp2",),
        call=ToolCall(
            call_id="dq3", args=CheckQualityArgs(series=("CA_2_FOODS",), days=period)
        ),
        result=ToolResult(
            call_id="dq3", ok=True, fields={"missing_dates": 4}, result_hash=HASH
        ),
    ),
    TraceStep(turn=4, summary="finish", observation_refs=("dq3",)),
)
packet = Packet(
    case_id="dev_001a",
    runner="agent",
    snapshot_id="snap_9f3c",
    terminal_state=TerminalState.STOP_UNTRUSTWORTHY,
    claims=(
        Claim(
            label=ClaimLabel.FINDING,
            template="{ev:dq3.missing_dates} actual dates are missing in the period.",
            row_refs=("r1201", "r1202", "r1203", "r1204"),
        ),
        Claim(
            label=ClaimLabel.INFERENCE,
            template="The MAE change is not comparable as computed.",
        ),
    ),
    not_established=("whether the model deteriorated",),
    trace=trace,
    cost=Cost(turns=4, calls=3, usd=0.04, seconds=11.2),
)
for line in packet.render():
    print(line)
assert packet.render()[0] == "4 actual dates are missing in the period."
assert Packet.model_validate_json(packet.model_dump_json()) == packet